In [62]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [34]:
df = pd.read_csv("spotify_2015_2025_85k.csv")

In [35]:
df.head(10)


,track_id,track_name,artist_name,album_name,release_date,genre,duration_ms,popularity,danceability,energy,key,loudness,mode,instrumentalness,tempo,stream_count,country,explicit,label
0,TRK-BEBD53DA84E1,Agent every (0),Noah Rhodes,Beautiful instead,2016-04-01,Pop,234194,55,0.15,0.74,9,-32.22,0,0.436,73.12,13000,Brazil,0,Universal Music
1,TRK-6A32496762D7,Night respond,Jennifer Cole,Table,2022-04-15,Metal,375706,45,0.44,0.46,0,-14.02,0,0.223,157.74,1000,France,1,Island Records
2,TRK-47AA7523463E,Future choice whatever,Brandon Davis,Page southern,2016-02-23,Rock,289191,55,0.62,0.80,8,-48.26,1,0.584,71.03,1000,Germany,1,XL Recordings
3,TRK-25ADA22E3B06,Bad fall pick those,Corey Jones,Spring,2015-10-12,Pop,209484,51,0.78,0.98,1,-34.47,1,0.684,149.00,1000,France,0,Warner Music
4,TRK-9245F2AD996A,Husband,Mark Diaz,Great prove,2022-07-08,Indie,127435,39,0.74,0.18,10,-17.84,0,0.304,155.85,2000,United States,0,Independent
5,TRK-A249E0859674,Move each,Devin Schaefer,Detail food,2018-01-01,Rock,200615,36,0.91,0.57,3,-19.61,1,0.574,183.86,1000,Australia,0,Warner Music
6,TRK-80176DE44638,Husband at,Latoya Robbins,Reality,2023-05-10,Country,372579,38,0.89,0.41,2,-27.49,0,0.175,165.81,1000,United Kingdom,0,Warner Music
7,TRK-75503F2042FD,Much section investment on gun,Brenda Snyder PhD,Claim,2015-09-18,Classical,290078,70,0.61,0.47,8,-8.52,0,0.201,155.24,21000,United Kingdom,0,Island Records
8,TRK-5BB824DBF141,His other,David Tran,Response,2019-11-16,Hip-Hop,327881,66,0.73,0.96,8,-5.72,1,0.610,177.83,6000,Japan,0,Independent
9,TRK-3EB4332CA819,Specific,Ashley Graham,About reveal,2021-01-18,Country,90299,40,0.61,0.49,5,-7.55,1,0.089,93.52,1000,Brazil,1,Sony Music


In [36]:
df.shape

(85000, 19)

In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85000 entries, 0 to 84999
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   track_id          85000 non-null  object 
 1   track_name        85000 non-null  object 
 2   artist_name       85000 non-null  object 
 3   album_name        85000 non-null  object 
 4   release_date      85000 non-null  object 
 5   genre             85000 non-null  object 
 6   duration_ms       85000 non-null  int64  
 7   popularity        85000 non-null  int64  
 8   danceability      85000 non-null  float64
 9   energy            85000 non-null  float64
 10  key               85000 non-null  int64  
 11  loudness          85000 non-null  float64
 12  mode              85000 non-null  int64  
 13  instrumentalness  85000 non-null  float64
 14  tempo             85000 non-null  float64
 15  stream_count      85000 non-null  int64  
 16  country           85000 non-null  object

In [40]:
#cleaning 

missing = df.isnull().sum()

missing #no missing values

track_id            0
track_name          0
artist_name         0
album_name          0
release_date        0
genre               0
duration_ms         0
popularity          0
danceability        0
energy              0
key                 0
loudness            0
mode                0
instrumentalness    0
tempo               0
stream_count        0
country             0
explicit            0
label               0
dtype: int64

In [42]:
rows_before = len(df)
df = df.dropna(subset=["track_name"])
print(f"\n  →{rows_before - len(df)} rows removed without track_name")


  →0 rows removed without track_name


In [46]:
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
bad_dates = df["release_date"].isnull().sum()
bad_dates

0

In [48]:
out_of_range_dates = df[
    (df["release_date"].dt.year < 2015) | (df["release_date"].dt.year > 2025)
]
print(f"  Out of range 2015-2025: {len(out_of_range_dates)} rows")
if len(out_of_range_dates) > 0:
    print(f"  {out_of_range_dates['release_date'].dt.year.value_counts()}")
    df = df[
        (df["release_date"].dt.year >= 2015) & (df["release_date"].dt.year <= 2025)
    ]
    print(f" -> {len(out_of_range_dates)} rows out of range")

  Out of range 2015-2025: 0 rows


In [50]:
df["explicit"] = df["explicit"].astype(bool)
df["mode"]     = df["mode"].astype(bool)

df["explicit"] # to observe if a song is appropriate or not

0        False
1         True
2         True
3        False
4        False
         ...  
84995    False
84996    False
84997    False
84998    False
84999    False
Name: explicit, Length: 85000, dtype: bool

In [54]:
dup_full     = df.duplicated().sum()

dup_full #no dups

0

In [55]:
dup_track_id = df["track_id"].duplicated().sum()
dup_track_id #no dups

0

In [59]:
#check to see out of range values
audio_feature_bounds = {
    "danceability"    : (0.0, 1.0),
    "energy"          : (0.0, 1.0),
    "instrumentalness": (0.0, 1.0),
    "loudness"        : (-60.0, 0.0),  # dB
    "tempo"           : (40.0, 250.0), # BPM
    "popularity"      : (0, 100),
    "key"             : (0, 11),
}

total_clipped = 0
for col, (lo, hi) in audio_feature_bounds.items():
    out_of_range = ((df[col] < lo) | (df[col] > hi)).sum()
    if out_of_range > 0:
        df[col] = df[col].clip(lower=lo, upper=hi)
        print(f"  {col:<20}: {out_of_range:>5} out of range values [{lo}, {hi}] → clip")
        total_clipped += out_of_range
    else:
        print(f"  {col:<20}: in range")

print(f"\n  Total out of range values: {total_clipped}")

  danceability        : in range
  energy              : in range
  instrumentalness    : in range
  loudness            : in range
  tempo               : in range
  popularity          : in range
  key                 : in range

  Total out of range values: 0


In [65]:
log_streams = np.log1p(df["stream_count"])  #searching outliers with IQR method
Q1, Q3 = log_streams.quantile([0.25, 0.75])
IQR = Q3 - Q1
lower_log = Q1 - 1.5 * IQR
upper_log = Q3 + 1.5 * IQR

stream_outliers = df[(log_streams < lower_log) | (log_streams > upper_log)]
print(f"  stream_count outliers (IQR on log scale): {len(stream_outliers)}")
print(f"   outliers range: [{stream_outliers['stream_count'].min():,.0f}"
      f", {stream_outliers['stream_count'].max():,.0f}]")

  stream_count outliers (IQR on log scale): 4152
   outliers range: [243,000, 20,000,000]


In [66]:
print(f"\n  duration_ms:")
print(f"    Min: {df['duration_ms'].min():,} ms  ({df['duration_ms'].min()/1000:.1f} sec)")
print(f"    Max: {df['duration_ms'].max():,} ms  ({df['duration_ms'].max()/1000:.1f} sec)")




  duration_ms:
    Min: 90,004 ms  (90.0 sec)
    Max: 420,000 ms  (420.0 sec)


In [67]:
# Conversion in seconds 
df["duration_sec"] = df["duration_ms"] / 1000
df["duration_min"] = (df["duration_ms"] / 60000).round(2)

In [68]:
OUTPUT_PATH = "spotify_cleaned.csv"

In [69]:
df.to_csv(OUTPUT_PATH, index=False)